# Measuring tune response ophyd async style

## import and set up

###  standard modules

In [ ]:
import logging


import yaml
import jsons
import pprint

additional modules for extra info

In [ ]:
import os
import pprint
import itertools

### logging level

In [ ]:
# avoid being messed up with communication debug messages
logging.basicConfig(level=logging.WARNING)


### accml and accml lib modules

In [ ]:
from accml.core.utils.basic_measurement_execution_engine import BasicMeasurementExecutionEngine
from accml.core.utils.simple_storage import SimpleDataStorage
from accml.custom.ophyd_async.ophyd_async_backend import OphydAsyncDeviceBackendRW
from accml.custom.ophyd_async.ophyd_async_measurement_execution_engine import OphydAsyncDeltaBackendRWProxy
from accml_lib.core.bl.command_rewritter import CommandRewriter
from accml_lib.core.bl.delta_backend import StateCache
from accml_lib.core.model.output.tune import Tune
from accml_lib.custom.bessyii.liasion_translator_setup import load_managers

from accml.app.tune.model import TuneResponseCollection

In [ ]:
from accml_lib.core.model.utils.command import Command, CommandSequence, ReadCommand, TransactionCommand, BehaviourOnError

### jsons for exporting data model

jsons takes care to export the datamodel into basic types (lists, dictonaries, int, string, float...)
the output can then be stored in appropriate formates eg. json or yaml

In [ ]:
from accml_lib.core.model.output.result import register_serializers_to_json_fork

jsons_fork = jsons.fork()
register_serializers_to_json_fork(jsons_fork)

## device setup and connection

### connection to devices

In [ ]:
from accml_lib.custom.bessyii.setup import setup

For the machine: typically no additional prefix

In [ ]:
devices = setup(prefix="")

for the twin:

  * give the prefix, don't forget to add a ":" if you use the Twin's defaukt
  * set it to None: then it will use its external mimic and evaluate the OS

In [ ]:
devices = setup(prefix=os.getenv("USER") +":")

In [ ]:
await devices.get("tune").connect()
await devices.get('quadrupole_pcs').connect()

### Manager setup

In [ ]:
yp, lm, ts = load_managers()

In [ ]:
pprint.pprint(yp)

In [ ]:
pprint.pprint(lm)

In [ ]:
pprint.pprint(ts)

### setup of communication

In [ ]:
backend = OphydAsyncDeviceBackendRW(devices=devices)

In [ ]:
state_cache=StateCache(name="BESSY2_OphydAsync_Dev_State_Cache")
delta_backend = OphydAsyncDeltaBackendRWProxy(
    backend=backend,
    cache=state_cache,
)

In [ ]:
mexec = BasicMeasurementExecutionEngine(
    backend=delta_backend,
    cmd_rewriter=CommandRewriter(liaison_manager=lm, translation_service=ts),
    storage=SimpleDataStorage(),
    expected_view_for_output="device"
)


## The measurement itself

### The commands

In [ ]:
measurement_values = [0, -1, 0, 1, 0]

In [ ]:
cmds_on_machine = []
for name in yp.get("quadrupole_pcs"):
    for val in measurement_values:
        cmds_on_machine.append(
            TransactionCommand(
                transaction=[Command(id=name, property="delta_set_current", value=val, behaviour_on_error=BehaviourOnError.stop)]
            )
        )
cmds_on_machine = CommandSequence(commands=cmds_on_machine)


In [ ]:
def device_identifers_in_commands(commands: Sequence[TransactionCommand]) -> Sequence[str]:
    return tuple(
        itertools.chain(*[
            [cmd.id for cmd in tc.transaction]
            for tc in cmds_on_machine.commands
        ])
    )

In [ ]:
device_identifers_in_commands(cmds_on_machine);

In [ ]:
def contains(tc: TransactionCommand, name: str) -> bool:
    for cmd in tc.transaction:
        if cmd.id == name:
            return True
    return False

sel = CommandSequence(
    commands=list(
        filter(lambda tc: contains(tc, "Q3PD1R"), cmds_on_machine.commands)
    )
)
pprint.pprint(sel)

I want to be sure that I exclude one power converter (the emil quadrupole to be precise)

In [ ]:
def not_contains(tc: TransactionCommand, name: str) -> bool:
    for cmd in tc.transaction:
        if cmd.id != name:
            return True
    return False

[name for name in yp.get("quadrupoles") if "QI" in name]    

In [ ]:
cmds_on_machine = CommandSequence(
    commands=list(
        filter(lambda tc: not_contains(tc, "QIT6R"), cmds_on_machine.commands)
    )
)
# pprint.pprint(sel)

In [ ]:
assert "Q1T6R" not  in  itertools.chain(*[
        [cmd.id for cmd in tc.transaction]
        for tc in cmds_on_machine.commands
    ])

## Taking data: a single value

### using backend directly

In [ ]:
await backend.trigger(dev_id="Q3PDR", prop_id="set_current")

In [ ]:
r = await backend.read(dev_id="Q3PD1R", prop_id="set_current")
r

In [ ]:
backend.devices._devices.keys()

### using measurement execution engine

In [ ]:
await mexec.trigger_read([ReadCommand(id="Q3PD1R", property="set_current")])

### now lets get under the hood: ophyd async

In [ ]:
backend.get_natural_view_name()

In [ ]:
dev = backend.devices.get("Q3PD1R")

In [ ]:
# await dev.trigger()

In [ ]:
await dev.read()

### Take full  measurement

In [ ]:
md = {}

uid = mexec.execute(
    detectors=detectors,
    commands_collection=cmds_on_machine.commands,  # need to add bpms
    # detectors=[tunes],
#    # actuators=actuators,
    # info_signals=info_signals,
    md=md,
)

uid,

### Analysis preparation: what happened  

#### reference point: inspect cache

In [ ]:
list(state_cache.cache)

In [ ]:
state_cache.get(ReadCommand(id="Q1M1D1R", property="set_current"))

In [ ]:
data = storage.get(uuid)
data = jsons.dump(data, fork_inst=jsons_fork)

with open("tune_response_data_from_simulator.json", "wt") as fp:
    json.dump(data, fp, indent=4)

